# January 2026 — decision-time wind forecast study

This notebook runs against the included genuine Elexon FUELHH WIND and WINDFOR snapshot. The sample is the **31 purchase decisions from 1–31 January**, with deliveries **2 January–1 February 2026**. The test uses hourly WINDFOR targets matched to FUELHH at the same timestamp; there are no fabricated half-hour forecasts.

**Question:** what forecast was available before the selected market gate, and how did that forecast compare with the subsequently recorded series?

In [1]:
from pathlib import Path
import sys, json
root = Path.cwd()
if not (root / 'backend').exists():
    root = root.parent
sys.path.insert(0, str(root / 'backend'))
from app.store import get_store
from app.planning import AUCTIONS, delivery_days, planned_day, month_rows, validation
from datetime import date
from statistics import mean
store = get_store()
assert store['available'] and store['source'] == 'ELEXON_FUELHH_WINDFOR'
assert len(store['actual']) == 1488 and len(store['forecast']) == 744
print('SOURCE:', store['source'])
print('Actual half-hour targets:', len(store['actual']))
print('Native hourly forecast target timestamps:', len(store['forecast']))
print('Forecast publication records:', sum(len(x) for x in store['forecast'].values()))

SOURCE: ELEXON_FUELHH_WINDFOR
Actual half-hour targets: 1488
Native hourly forecast target timestamps: 744
Forecast publication records: 17864


## Auction comparisons

Each hourly target selects the latest publication no later than **exchange gate minus an assumed five-minute margin**. The error is WINDFOR minus actual FUELHH; positive means the forecast series was higher.

In [2]:
for name, a in AUCTIONS.items():
    rows = [r for day in month_rows(name, store).values() for r in day if r['error'] is not None]
    errors = [r['error'] for r in rows]
    raw_mae = mean(abs(e) for e in errors)
    print(f"{a['label']:19s} {a['cutoff']} | compared={len(rows):3d} | MAE={raw_mae:,.0f} MW | bias={mean(errors):+,.0f} MW")

EPEX hourly         09:20 | compared=744 | MAE=2,022 MW | bias=+1,746 MW
N2EX hourly         09:50 | compared=744 | MAE=1,996 MW | bias=+1,728 MW
EPEX half-hourly    15:30 | compared=744 | MAE=1,886 MW | bias=+1,691 MW


## Chronological walk-forward, not leave-one-day-out

The adjustment is fitted to **completed delivery days on or before D−2**, and never to the selected day or subsequent dates. We require seven prior complete delivery days. Results therefore cover fewer than 744 observations. These data could still include retrospective revisions to earlier actuals, because only the final available FUELHH snapshot was provided.

In [3]:
for name, a in AUCTIONS.items():
    v = validation(name, store)
    print(f"{a['label']:19s} n={v['n']} | raw MAE={v['rawMaeMW']:,} MW | level-adjusted MAE={v['levelMaeMW']:,} MW | lower reference observed={v['aboveFloorPct']}%")

EPEX hourly         n=552 | raw MAE=2,123 MW | level-adjusted MAE=1,382 MW | lower reference observed=85.5%
N2EX hourly         n=552 | raw MAE=2,103 MW | level-adjusted MAE=1,352 MW | lower reference observed=85.9%
EPEX half-hourly    n=552 | raw MAE=1,979 MW | level-adjusted MAE=1,197 MW | lower reference observed=85.0%


## One reconstructed decision: 20 January → 21 January delivery

In [4]:
d = date(2026, 1, 21)
rows, fits, train_days = planned_day(d, 'epex-hourly', store)
assert train_days == 18
assert all(r['publishTime'] < '2026-01-20T09:20:00Z' for r in rows)
print('Earlier complete delivery days used:', train_days)
print('Mean published WINDFOR:', round(mean(r['windfor'] for r in rows)), 'MW')
print('Mean exploratory adjusted value:', round(mean(r['plan'] for r in rows)), 'MW')
print('Mean FUELHH actual:', round(mean(r['actual'] for r in rows)), 'MW')
print('Historical lower reference, mean:', round(mean(r['low'] for r in rows)), 'MW')

Earlier complete delivery days used: 18
Mean published WINDFOR: 19390 MW
Mean exploratory adjusted value: 16529 MW
Mean FUELHH actual: 15713 MW
Historical lower reference, mean: 13829 MW


## Interpretation and limits

The selected WINDFOR series is above the matched FUELHH series in many hours, especially when the forecast is high. The training period is short, and there may be a difference in population coverage between the two datasets. The notebook does **not** prove curtailment, model bias, the quality of an operational procurement strategy, or a guaranteed future confidence interval. Demand, market prices and positions are outside the assignment. The dashboard reflects these limits in its wording.